# 04 — Otimização de Portfólio, Guardrails Operacionais e Validação de -R$ 10.000.000,00

**Objetivo:** Verificar linha por linha a equação contábil (`Impacto Líquido OM = MC Incremental - Custo Bruto - Custo Op./SLA`), comprovar o fechamento exato em **-R$ 10.000.000,00** e validar os parâmetros técnicos de **City 17** (capacidade logística) e **City 6** (folga de 0,46 p.p. até 98%).


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

PROJECT_DIR = "/content/drive/MyDrive/ifood-growth-competition-case"
if os.path.exists("/content/drive"):
    if not os.path.exists(PROJECT_DIR):
        from google.colab import drive
        drive.mount("/content/drive")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        PROJECT_DIR = os.getcwd()

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

for sub in ["data/raw", "data/processed", "outputs/excel", "outputs/figures", "outputs/reports", "dashboard"]:
    os.makedirs(sub, exist_ok=True)

print("Diretório ativo:", os.getcwd())


In [ ]:
df_alloc = pd.read_csv("data/processed/base_alocacao_otimizada.csv")

# Validação linha por linha
df_alloc["ped_liq_check"] = (df_alloc["pedidos_brutos"] * (1.0 - df_alloc["canibalizacao_pct"] / 100.0)).round(2)
df_alloc["om_check"] = (df_alloc["mc_incremental_rs"] - df_alloc["custo_bruto_rs"] - df_alloc["custo_operacional_sla_rs"]).round(2)
df_alloc["diff_om"] = (df_alloc["om_check"] - df_alloc["impacto_liquido_om_rs"]).round(2)

assert (df_alloc["diff_om"] == 0).all(), "Divergência encontrada na soma linha por linha!"
soma_om = round(df_alloc["impacto_liquido_om_rs"].sum(), 2)
assert soma_om == -10000000.00, f"Esperado -10000000.00, obtido {soma_om}"

print("VALIDAÇÃO CONTÁBIL APROVADA — SOMA ALGÉBRICA = R$", f"{soma_om:,.2f}")
display(df_alloc[[
    "cidade", "classificacao", "alavanca", "custo_bruto_rs", "pedidos_brutos",
    "canibalizacao_pct", "pedidos_incrementais_liquidos", "gmv_incremental_rs",
    "mc_incremental_rs", "custo_operacional_sla_rs", "impacto_liquido_om_rs", "diff_om"
]])

# Detalhamento técnico City 17 e City 6
r17 = df_alloc[df_alloc["cidade"] == "City 17"].iloc[0]
print("\n=== CITY 17 — CAPACIDADE LOGÍSTICA ===")
print(f"Capacidade base:            {r17['capacidade_base_pedidos_equiv']:,.2f} pedidos equivalentes")
print(f"Capacidade adicional:       +{r17['capacidade_adicional_pedidos_equiv']:,.2f} pedidos equivalentes (+{r17['capacidade_adicional_pct']:.2f}%)")
print(f"Saturação base:             {r17['saturacao_base_pct']:.2f}%")
print(f"Saturação sem intervenção:  {r17['saturacao_sem_intervencao_pct']:.2f}%")
print(f"Saturação pós-investimento: {r17['saturacao_pos_investimento_pct']:.2f}% (redução de {r17['saturacao_pos_investimento_pct'] - r17['saturacao_sem_intervencao_pct']:.2f} p.p. vs. sem intervenção)")

r6 = df_alloc[df_alloc["cidade"] == "City 6"].iloc[0]
print("\n=== CITY 6 — GUARDRAIL DE SATURAÇÃO ===")
print(f"Saturação base -> pós-inv.: {r6['saturacao_base_pct']:.2f}% -> {r6['saturacao_pos_investimento_pct']:.2f}%")
print(f"Folga até guardrail 98%:    {r6['folga_guardrail_98_pp']:.2f} p.p.")
print(f"Custo op./SLA incremental:  R$ {r6['custo_operacional_sla_rs']:,.2f} (8% custo bruto + R$ 25.000,00 SLA extra)")
